In [0]:
from pyspark.sql.functions import current_timestamp

# Passo 1 - verifica se gold existe, pega MAX
if spark.catalog.tableExists("oc_projeto.`oc-tabelas`.gold_carga_staff"):
    max_ts = spark.sql("""
        SELECT max(data_processamento) as ts FROM oc_projeto.`oc-tabelas`.gold_carga_staff
    """).collect()[0]["ts"]
else:
    max_ts = None

# Passo 2 - filtra a silver
if max_ts is None:
    df_gold_novo = spark.table("oc_projeto.`oc-tabelas`.silver_voos_dep")
else:
    df_gold_novo = spark.table("oc_projeto.`oc-tabelas`.silver_voos_dep").filter(f"data_processamento > '{max_ts}'")



df_gold_novo = df_gold_novo.withColumn("data_process_gold", current_timestamp())

# Passo 3 - cria a view ANTES de rodar a query de dedup
df_gold_novo.createOrReplaceTempView("temp_gold_dep_novo")



In [0]:
df_enriquecido = spark.sql("""
    WITH temp_dedup AS (
        SELECT
            voo_dep,
            iata_dep,
            data,
            first(peso_carga_inicial) as peso_carga_inicial,
            first(peso_carga_final) as peso_carga_final,
            first(peso_bagagem_inicial) as peso_bagagem_inicial,
            first(peso_bagagem_final) as peso_bagagem_final,
            first(bag_qtd) as bag_qtd,
            first(staff_qtd) as staff_qtd,
            first(motivo_corte) as motivo_corte,
            first(data_processamento) as data_processamento,
            first(data_process_gold) as data_process_gold
        FROM temp_gold_dep_novo
        GROUP BY voo_dep, iata_dep, data
    )
    SELECT
        voo_dep,
        iata_dep,
        data,
        peso_carga_inicial as pci,
        peso_carga_final as pcf,
        peso_bagagem_inicial as pbi,
        peso_bagagem_final as pbf,
        bag_qtd as bgt,
        staff_qtd as staff,
        --motivo_corte,
        CASE
            WHEN peso_carga_inicial IS NULL OR peso_carga_final IS NULL THEN 0
            ELSE CAST(peso_carga_inicial - peso_carga_final AS BIGINT)
        END as peso_corte,
        CASE
            WHEN peso_carga_inicial - peso_carga_final = 0 THEN 'OK'
            WHEN peso_carga_inicial-peso_carga_final > 700 and staff < 4 THEN 'efetivo'
            WHEN peso_carga_inicial-peso_carga_final < 700 and staff >= 4 THEN 'GerSwp'
            WHEN peso_carga_inicial-peso_carga_final > 700 and staff >= 4 THEN 'GerSwp'
            WHEN peso_carga_inicial-peso_carga_final > 700 and staff IS NULL THEN 'efetivo'
            WHEN peso_carga_inicial-peso_carga_final > 0 AND peso_carga_inicial-peso_carga_final < 100 THEN 'cubagem'
            WHEN motivo_corte IS NULL THEN 'OK'
            ELSE motivo_corte
        END as motivo_status,
        data_processamento,
        data_process_gold
    FROM temp_dedup
""")

In [0]:
if not spark.catalog.tableExists("oc_projeto.`oc-tabelas`.gold_carga_staff"):
    df_enriquecido.write.format("delta").saveAsTable("oc_projeto.`oc-tabelas`.gold_carga_staff")
else:
    df_enriquecido.createOrReplaceTempView("temp_gold_carga_staff")

    spark.sql("""
          MERGE INTO oc_projeto.`oc-tabelas`.gold_carga_staff AS t
          USING temp_gold_carga_staff AS s
          ON t.voo_dep = s.voo_dep AND t.iata_dep = s.iata_dep AND t.data = s.data
          WHEN NOT MATCHED THEN INSERT *
          """)

print("Dados carregados com sucesso na gold_carga_staff com MERGE")